# EDA — datasets oficiais

Análise da base de treino e teste do classificador. Ela junta as bases escolhidas
depois das análises anteriores (`notebooks/eda.ipynb` e `doc/eda.md` registram por que o
Fakepedia e o texto do FakeRecogna ficaram de fora).

| `base` | Classe | O que é | Período |
|---|---|---|---|
| `fakebr` | fake e true | Fake.br-Corpus (NILC/USP): pares de notícias falsas e verdadeiras sobre o mesmo assunto | 2016–2018 |
| `fakerecogna` | fake e true | FakeRecogna (UNESP) com o texto recoletado: boatos do Boatos.org e matérias de G1, UOL e Extra | 2019–2021 |
| `boatos` | fake | Boatos.org, coletado por nós: o boato como circulou (mensagem citada ou alegação) | 2024–2026 |
| `factcheck` | fake | Google Fact Check Tools API: alegações com veredito "falso" de 10 agências brasileiras | 2018–2026 |
| `fakenewsbr` | fake | FakenewsBR v6: só as sub-bases brasileiras, originais e com veredito "falso" | 2018–2026 |
| `verdadeiras` | true | CSV próprio: matérias de 9 portais de linhas editoriais variadas | 2018–2026 |

**Regras aplicadas no preparo** (`scripts/preparar_dados.py`):
- **Rótulo fake:** só vereditos "falso", "fake", "montagem" e "predominantemente falso".
  Meias-verdades (enganoso, fora de contexto, distorcido…) e sátira ficam de fora.
- **Fora da FakenewsBR:** traduções do inglês, português de Portugal, texto de checador e
  a sub-base do Kaggle sem licença.
- **Portais das verdadeiras:** só os que **não** bloqueiam robôs de IA no `robots.txt`.
- **Duplicatas:** mesma URL ou mesmo início de texto entram uma vez só. Quase-duplicatas
  (o mesmo boato reescrito) de um registro de treino ou validação saem dos splits
  posteriores e viram `fora`.
- **Dois protocolos de divisão**, nas colunas `split` e `split_produto`:

| | Protocolo A, "referência" (`split`) | Protocolo B, "produto" (`split_produto`) |
|---|---|---|
| Treino | Fake.br + FakeRecogna (10% para validação) | Fake.br + FakeRecogna + as bases novas de 2016 a 2023, menos as fontes deixadas de fora |
| Validação | sorteio de 10% do treino | 2024, das mesmas fontes do treino |
| Teste | todo o resto | `teste_temporal`: 2025 em diante, das fontes vistas no treino · `teste_fontes`: AFP Checamos, Aos Fatos, COVID19.BR, Veja, Agência Pública e Jovem Pan, de qualquer ano · `fora`: sem data ou anterior a 2016 (exceto o Fake.br, que é pareado) |
| Responde | quanto as bases públicas generalizam para fontes novas | quão bom é o modelo que vai para o jogo, em fontes e períodos que ele não viu |

Cada notícia tem `texto` (a versão mais longa) e `texto_curto` (alegação ou título),
que é o formato das perguntas do jogo.

In [ ]:
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CORES = {"fake": "#eb6834", "true": "#2a78d6"}
TINTA_2 = "#52514e"
BASES = ["fakebr", "boatos", "factcheck", "fakenewsbr", "verdadeiras"]
PROTOCOLOS = {
    "split": ("A (referência)", ["treino", "validacao", "teste"]),
    "split_produto": ("B (produto)", ["treino", "validacao", "teste_temporal", "teste_fontes"]),
}
plt.rcParams.update({
    "figure.dpi": 110, "figure.figsize": (9, 4), "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.color": "#e6e5e0",
    "grid.linewidth": 0.8, "axes.axisbelow": True, "axes.edgecolor": "#b8b7b0",
    "axes.labelcolor": TINTA_2, "xtick.color": TINTA_2, "ytick.color": TINTA_2,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False,
})
pd.set_option("display.max_colwidth", 110)

df = pd.read_json(RAIZ / "data" / "processed" / "dataset.jsonl", lines=True, dtype={"par_id": "Int64"})
df = df[df.rotulo.notna()]  # análise binária: os enganosos (veracidade de 3 níveis) ficam de fora
df["data"] = pd.to_datetime(df["data"], errors="coerce")
df["ano"] = df.data.dt.year.astype("Int64")
df["n_palavras"] = df.texto.str.split().str.len()
df["curto"] = df.texto_curto.fillna("")
df["n_palavras_curto"] = df.curto.str.split().str.len()
print(f"{len(df):,} notícias".replace(",", "."))

## 1. Composição e equilíbrio

In [ ]:
for coluna, (nome, ordem) in PROTOCOLOS.items():
    comp = df.groupby([coluna, "base", "rotulo"]).size().unstack(fill_value=0).reindex(ordem, level=0)
    print(f"Protocolo {nome}")
    display(comp.astype(int))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4), sharex=True)
resumos = []
for ax, (coluna, (nome, ordem)) in zip(axes, PROTOCOLOS.items()):
    por_split = df.groupby([coluna, "rotulo"]).size().unstack(fill_value=0).reindex(ordem).fillna(0)
    esquerda = np.zeros(len(por_split))
    for rotulo in ["true", "fake"]:
        ax.barh(por_split.index, por_split[rotulo], left=esquerda, color=CORES[rotulo], height=0.55,
                label=rotulo, edgecolor="white", linewidth=2)
        esquerda += por_split[rotulo].values
    ax.invert_yaxis()
    ax.set_title(f"Protocolo {nome}")
    ax.set_xlabel("notícias")
    ax.grid(axis="y", visible=False)
    por_split["% fake"] = (100 * por_split.fake / por_split[["fake", "true"]].sum(axis=1)).round(1)
    resumos.append(por_split.assign(protocolo=nome).set_index("protocolo", append=True))
axes[0].legend(loc="lower right")
fig.tight_layout()
pd.concat(resumos)

## 2. De onde vêm as falsas

In [ ]:
falsas = df[df.rotulo == "fake"]
(falsas.groupby("base").fonte.value_counts().groupby(level=0).head(6)
 .rename("notícias").to_frame().reindex(BASES, level=0))

In [ ]:
top_fontes = falsas.fonte.value_counts()
fig, ax = plt.subplots(figsize=(8, 5))
top = top_fontes.head(15)[::-1]
ax.barh(top.index, top.values, color=CORES["fake"], height=0.6)
ax.set_title("Falsas: as 15 fontes com mais notícias (após deduplicação)")
ax.set_xlabel("notícias")
ax.grid(axis="y", visible=False)
print(f"{falsas.fonte.nunique()} fontes distintas; a maior responde por "
      f"{100 * top_fontes.iloc[0] / len(falsas):.1f}% das falsas")

## 3. De onde vêm as verdadeiras

In [ ]:
verd = df[df.base == "verdadeiras"]
tabela_portais = verd.pivot_table(index=["categoria", "fonte"], columns="ano", values="id", aggfunc="size", fill_value=0)
tabela_portais["total"] = tabela_portais.sum(axis=1)
tabela_portais

In [ ]:
(verd.groupby("split_produto").categoria.value_counts(normalize=True).mul(100).round(1)
 .unstack().reindex(PROTOCOLOS["split_produto"][1]).rename_axis(columns="linha editorial (%), protocolo B"))

A coleta pega o mesmo número de matérias por portal e por janela de ~5 dias, então a
mistura de linhas editoriais fica estável entre os anos. Um portal com menos matérias num
ano indica que ele publicava menos (ou não existia) naquele período. No protocolo B, o
`teste_fontes` tem só os três portais deixados de fora (Veja, Agência Pública e Jovem
Pan), um por linha editorial. No protocolo A, todas as verdadeiras dos portais são teste.

## 4. Período

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
com_data = df.dropna(subset=["data"])
com_data = com_data[com_data.data >= "2015-01-01"]
for rotulo, g in com_data.groupby("rotulo"):
    serie = g.set_index("data").resample("QS").size()
    ax.plot(serie.index, serie.values, color=CORES[rotulo], lw=2, label=rotulo)
for corte, nome in [("2024-01-01", "validação"), ("2025-01-01", "teste")]:
    ax.axvline(pd.Timestamp(corte), color="#b8b7b0", lw=1, ls="--")
    ax.annotate(nome, (pd.Timestamp(corte), ax.get_ylim()[1] * 0.92), xytext=(4, 0),
                textcoords="offset points", color=TINTA_2, fontsize=9)
ax.set_title("Notícias por trimestre")
ax.set_ylabel("notícias")
ax.legend(loc="upper left")
print("sem data:", df.data.isna().groupby(df.base).sum().to_dict())

Falsas e verdadeiras precisam cobrir os **mesmos períodos**, senão a época vira atalho:
o modelo associaria "Covid" ou "eleição de 2022" a uma classe só porque uma delas tinha
mais notícias daquela época.

## 5. Qualidade

In [ ]:
qualidade = pd.DataFrame({
    "notícias": df.groupby(["base", "rotulo"]).size(),
    "sem texto_curto (%)": df.assign(x=df.curto == "").groupby(["base", "rotulo"]).x.mean() * 100,
    "texto_curto < 4 palavras (%)": df.assign(x=(df.curto != "") & (df.n_palavras_curto < 4)).groupby(["base", "rotulo"]).x.mean() * 100,
    "texto < 10 palavras (%)": df.assign(x=df.n_palavras < 10).groupby(["base", "rotulo"]).x.mean() * 100,
}).round(1).reindex(BASES, level=0)
qualidade

In [ ]:
# Quase-duplicatas entre splits: o mesmo boato costuma ser checado de novo anos depois.
chave = df.curto.where(df.curto != "", df.texto).str.lower().str.replace(r"[^a-z0-9à-ü]+", "", regex=True).str[:80]
for coluna, (nome, ordem) in PROTOCOLOS.items():
    treino_chaves = set(chave[df[coluna] == "treino"])
    print(f"Protocolo {nome}")
    for split in ordem[1:]:
        mask = df[coluna] == split
        print(f"  {split}: {chave[mask].isin(treino_chaves).sum()} de {mask.sum()} começam igual a uma notícia do treino")

## 6. Formato: alegação curta × título

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
bins_curto = np.arange(0, 45, 1)
for rotulo in ["true", "fake"]:
    s = df[(df.rotulo == rotulo) & (df.curto != "")].n_palavras_curto
    axes[0].hist(s.clip(upper=44), bins=bins_curto, color=CORES[rotulo], alpha=0.6, label=rotulo, density=True)
axes[0].set_title("texto_curto: palavras")
axes[0].set_xlabel("palavras (44+ agrupadas)")
axes[0].legend()
bins_longo = np.logspace(0.5, 4, 40)
for rotulo in ["true", "fake"]:
    axes[1].hist(df[df.rotulo == rotulo].n_palavras, bins=bins_longo, color=CORES[rotulo], alpha=0.6, label=rotulo, density=True)
axes[1].set_xscale("log")
axes[1].set_title("texto: palavras (escala log)")
axes[1].set_xlabel("palavras")
fig.tight_layout()
(df[df.curto != ""].groupby(["base", "rotulo"]).n_palavras_curto.describe()[["count", "25%", "50%", "75%"]]
 .round(0).reindex(BASES, level=0))

O `texto` longo mistura formatos muito diferentes (mensagem de WhatsApp, alegação de uma
linha, matéria completa) e é dominado pelo tamanho. O `texto_curto` compara coisas
parecidas: uma frase afirmativa de cada lado.

## 7. Atalhos de estilo no texto curto

In [ ]:
EMOJI = re.compile("[\U0001F300-\U0001FAFF\u2600-\u27BF]")
def sinais(texto):
    palavras = re.findall(r"\w+", texto)
    n = max(len(palavras), 1)
    return {
        "palavras": len(palavras),
        "% CAIXA ALTA": 100 * sum(p.isupper() and len(p) > 1 for p in palavras) / n,
        "tem !": "!" in texto,
        "tem ?": "?" in texto,
        "tem aspas": bool(re.search("[\"“”‘’\']", texto)),
        "tem dois-pontos": ":" in texto,
        "tem número": bool(re.search(r"\d", texto)),
        "tem emoji": bool(EMOJI.search(texto)),
        "termina com ponto": texto.rstrip().endswith("."),
        "começa com maiúscula": texto[:1].isupper(),
    }
curtos = df[df.curto != ""].copy()
S = pd.DataFrame([sinais(t) for t in curtos.curto], index=curtos.index)
tabela_sinais = S.groupby(curtos.rotulo).mean()
for c in tabela_sinais.columns:
    if S[c].dtype == bool:
        tabela_sinais[c] *= 100
tabela_sinais.T.round(1)

In [ ]:
primeira = curtos.curto.str.extract(r"^\W*(\w+)")[0].str.lower()
top_primeiras = pd.concat({
    rotulo: primeira[curtos.rotulo == rotulo].value_counts(normalize=True).mul(100).round(1).head(12).reset_index()
    for rotulo in ["fake", "true"]
}, axis=1)
top_primeiras.columns = ["fake: palavra", "fake: %", "true: palavra", "true: %"]
top_primeiras

**Teste de atalho:** um classificador que só vê os sinais acima (tamanho, pontuação,
caixa alta, primeira palavra) e **nenhum conteúdo**. Ele treina no `treino` de cada
protocolo e é avaliado nos demais splits. Se a AUC for alta, o classificador de verdade
também vai aprender estilo de fonte em vez de veracidade. A queda entre validação e teste
mostra o quanto esse estilo é específico das fontes do treino.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

X = S.astype(float).join(pd.get_dummies(primeira.where(primeira.isin(primeira.value_counts().head(30).index), "outra"),
                                        prefix="1a").astype(float))
y = (curtos.rotulo == "fake").astype(int)
resultados, pesos_b = [], None
for coluna, (nome, ordem) in PROTOCOLOS.items():
    treino = curtos[coluna] == "treino"
    if y[treino].nunique() < 2:
        print(f"Protocolo {nome}: o treino ainda não tem as duas classes no texto_curto.")
        continue
    escala = StandardScaler().fit(X[treino])
    modelo = LogisticRegression(max_iter=2000).fit(escala.transform(X[treino]), y[treino])
    for split in ordem[1:]:
        mask = curtos[coluna] == split
        auc = (roc_auc_score(y[mask], modelo.predict_proba(escala.transform(X[mask]))[:, 1])
               if y[mask].nunique() == 2 else np.nan)
        resultados.append({"protocolo": nome, "avaliado em": split, "notícias": int(mask.sum()), "AUC só com estilo": auc})
    if coluna == "split_produto":
        pesos_b = pd.Series(modelo.coef_[0], index=X.columns).sort_values()
print("0,5 = acaso; 1,0 = separa tudo só pelo estilo")
display(pd.DataFrame(resultados).round(3))
if pesos_b is not None:
    print("Sinais que mais pesam (protocolo B):")
    display(pd.concat({"puxa para TRUE": pesos_b.head(8).round(2), "puxa para FAKE": pesos_b.tail(8)[::-1].round(2)}, axis=1))

## 8. Vocabulário característico do texto curto

In [ ]:
TOKEN = re.compile(r"[a-záàâãéêíóôõúüç]+")
def contar(textos):
    c = Counter()
    for t in textos:
        c.update(TOKEN.findall(t.lower()))
    return c

def log_odds(ca, cb, alpha=0.01, min_freq=30):
    fundo = ca + cb
    a0 = alpha * sum(fundo.values())
    na, nb = sum(ca.values()), sum(cb.values())
    linhas = []
    for w, f in fundo.items():
        if f < min_freq:
            continue
        aw = alpha * f
        ya, yb = ca[w], cb[w]
        delta = np.log((ya + aw) / (na + a0 - ya - aw)) - np.log((yb + aw) / (nb + a0 - yb - aw))
        linhas.append((w, delta / np.sqrt(1 / (ya + aw) + 1 / (yb + aw))))
    return pd.DataFrame(linhas, columns=["palavra", "z"]).set_index("palavra")

lo = log_odds(contar(curtos[curtos.rotulo == "fake"].curto), contar(curtos[curtos.rotulo == "true"].curto))
pd.concat({
    "mais típicas de FAKE": lo.nlargest(25, "z").reset_index(),
    "mais típicas de TRUE": lo.nsmallest(25, "z").reset_index(),
}, axis=1).round(1)

## 9. Assuntos de cada período (falsas)

In [ ]:
periodos = {"até 2019": falsas.ano <= 2019, "2020–2021": falsas.ano.between(2020, 2021),
            "2022–2023": falsas.ano.between(2022, 2023), "2024 em diante": falsas.ano >= 2024}
textos_falsas = falsas.texto_curto.fillna(falsas.texto)
resto = contar(textos_falsas)
temas = {}
for nome, mask in periodos.items():
    mask = mask.fillna(False)
    c = contar(textos_falsas[mask])
    lo_p = log_odds(c, resto - c, min_freq=20)
    temas[nome] = lo_p.nlargest(15, "z").index.tolist()
pd.DataFrame(temas)

Cada período tem seus assuntos: Covid em 2020–2021, eleições e urnas em 2022, e o que
aparecer de 2024 em diante. Como o teste é temporal, o modelo será avaliado em assuntos
que pouco viu no treino, e é isso que mede se ele generaliza.

## 10. Conclusões

_(preenchidas após rodar com os dados completos)_